# 创建自己的数据集

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]
!apt install git-lfs

You will need to setup git, adapt your email and name in the following cell.

In [ ]:
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"

You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
!pip install requests

In [2]:
import requests

url = "https://api.github.com/repos/huggingface/datasets/issues?page=1&per_page=1"
response = requests.get(url)

In [13]:
response.status_code

200

In [14]:
response.json()

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/8055',
  'repository_url': 'https://api.github.com/repos/huggingface/datasets',
  'labels_url': 'https://api.github.com/repos/huggingface/datasets/issues/8055/labels{/name}',
  'comments_url': 'https://api.github.com/repos/huggingface/datasets/issues/8055/comments',
  'events_url': 'https://api.github.com/repos/huggingface/datasets/issues/8055/events',
  'html_url': 'https://github.com/huggingface/datasets/pull/8055',
  'id': 4042507479,
  'node_id': 'PR_kwDODunzps7I6knM',
  'number': 8055,
  'title': 'feat: add 3D mesh support and MeshFolder builder',
  'user': {'login': 'Vinay-Umrethe',
   'id': 175500353,
   'node_id': 'U_kgDOCnXsQQ',
   'avatar_url': 'https://avatars.githubusercontent.com/u/175500353?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/Vinay-Umrethe',
   'html_url': 'https://github.com/Vinay-Umrethe',
   'followers_url': 'https://api.github.com/users/Vinay-Umrethe/followers',
   'foll

In [3]:
import os
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")  # Copy your GitHub token here
headers = {"Authorization": f"token {GITHUB_TOKEN}"}

In [4]:
import time
import math
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm


def fetch_issues(
    owner="huggingface",
    repo="datasets",
    num_issues=10_000,
    rate_limit=5_000,
    issues_path=Path("."),
):
    if not issues_path.is_dir():
        issues_path.mkdir(exist_ok=True)

    batch = []
    all_issues = []
    per_page = 100  # Number of issues to return per page
    num_pages = math.ceil(num_issues / per_page)
    base_url = "https://api.github.com/repos"

    for page in tqdm(range(num_pages)):
        # Query with state=all to get both open and closed issues
        query = f"issues?page={page}&per_page={per_page}&state=all"
        issues = requests.get(f"{base_url}/{owner}/{repo}/{query}", headers=headers)
        batch.extend(issues.json())

        if len(batch) > rate_limit and len(all_issues) < num_issues:
            all_issues.extend(batch)
            batch = []  # Flush batch for next time period
            print(f"Reached GitHub rate limit. Sleeping for one hour ...")
            time.sleep(60 * 60 + 1)

    all_issues.extend(batch)
    df = pd.DataFrame.from_records(all_issues)
    df.to_json(f"{issues_path}/{repo}-issues.jsonl", orient="records", lines=True)
    print(
        f"Downloaded all the issues for {repo}! Dataset stored at {issues_path}/{repo}-issues.jsonl"
    )

In [17]:
# Depending on your internet connection, this can take several minutes to run...
fetch_issues(rate_limit = 20_000)

  0%|          | 0/100 [00:00<?, ?it/s]

Downloaded all the issues for datasets! Dataset stored at ./datasets-issues.jsonl


In [11]:
from datasets import load_dataset

# 默认 chunksize=10MB，schema 从第一批推断。若前批 closed_at 全为 null，
# 后批出现真实时间戳就会报类型冲突。
# 将 chunksize 设为超过文件大小（~39MB），强制一次读完所有行，schema 推断自然正确。
issues_dataset = load_dataset(
    "json",
    data_files="datasets-issues.jsonl",
    split="train",
    chunksize=30 << 20,  # 30MB可以完整推断字段类型
)
issues_dataset

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment'],
    num_rows: 7980
})

In [6]:
sample = issues_dataset.shuffle(seed=666).select(range(5))

# Print out the URL and pull request entries
for url, pr in zip(sample["html_url"], sample["pull_request"]):
    print(f">> URL: {url}")
    print(f">> Pull request: {pr}\n")

>> URL: https://github.com/huggingface/datasets/issues/590
>> Pull request: None

>> URL: https://github.com/huggingface/datasets/issues/1622
>> Pull request: None

>> URL: https://github.com/huggingface/datasets/issues/5601
>> Pull request: None

>> URL: https://github.com/huggingface/datasets/pull/6779
>> Pull request: {'url': 'https://api.github.com/repos/huggingface/datasets/pulls/6779', 'html_url': 'https://github.com/huggingface/datasets/pull/6779', 'diff_url': 'https://github.com/huggingface/datasets/pull/6779.diff', 'patch_url': 'https://github.com/huggingface/datasets/pull/6779.patch', 'merged_at': datetime.datetime(2024, 4, 8, 13, 27, 43)}

>> URL: https://github.com/huggingface/datasets/issues/5905
>> Pull request: None



In [7]:
issues_dataset = issues_dataset.map(
    lambda x: {"is_pull_request": False if x["pull_request"] is None else True}
)

In [8]:
sample = issues_dataset.shuffle(seed=666).select(range(5))

# Print out the URL and pull request entries
for url, pr, is_pr in zip(sample["html_url"], sample["pull_request"], sample['is_pull_request']):
    print(f">> URL: {url}")
    print(f">> Pull request: {pr}")
    print(f'>> Is pull request: {is_pr}\n')

>> URL: https://github.com/huggingface/datasets/issues/590
>> Pull request: None
>> Is pull request: False

>> URL: https://github.com/huggingface/datasets/issues/1622
>> Pull request: None
>> Is pull request: False

>> URL: https://github.com/huggingface/datasets/issues/5601
>> Pull request: None
>> Is pull request: False

>> URL: https://github.com/huggingface/datasets/pull/6779
>> Pull request: {'url': 'https://api.github.com/repos/huggingface/datasets/pulls/6779', 'html_url': 'https://github.com/huggingface/datasets/pull/6779', 'diff_url': 'https://github.com/huggingface/datasets/pull/6779.diff', 'patch_url': 'https://github.com/huggingface/datasets/pull/6779.patch', 'merged_at': datetime.datetime(2024, 4, 8, 13, 27, 43)}
>> Is pull request: True

>> URL: https://github.com/huggingface/datasets/issues/5905
>> Pull request: None
>> Is pull request: False



In [30]:
issue_number = 2792
url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
response = requests.get(url, headers=headers)
response.json()

{'message': 'API rate limit exceeded for user ID 206844284. If you reach out to GitHub Support for help, please include the request ID D3FC:90D44:108B47:121571:69B26997 and timestamp 2026-03-12 07:21:59 UTC. For more on scraping GitHub and how it may affect your rights, please review our Terms of Service (https://docs.github.com/en/site-policy/github-terms/github-terms-of-service)',
 'documentation_url': 'https://docs.github.com/en/rest/using-the-rest-api/getting-started-with-the-rest-api#rate-limiting',
 'status': '403'}

In [40]:
def get_comments(issue_number):
    url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
    response = requests.get(url, headers=headers)
    return [r["body"] for r in response.json()]


# Test our function works as expected
get_comments(2792)

["@albertvillanova my tests are failing here:\r\n```\r\ndataset_name = 'gooaq'\r\n\r\n    def test_load_dataset(self, dataset_name):\r\n        configs = self.dataset_tester.load_all_configs(dataset_name, is_local=True)[:1]\r\n>       self.dataset_tester.check_load_dataset(dataset_name, configs, is_local=True, use_local_dummy_data=True)\r\n\r\ntests/test_dataset_common.py:234: \r\n_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ \r\ntests/test_dataset_common.py:187: in check_load_dataset\r\n    self.parent.assertTrue(len(dataset[split]) > 0)\r\nE   AssertionError: False is not true\r\n```\r\nWhen I try loading dataset on local machine it works fine. Any suggestions on how can I avoid this error?",
 'Thanks for the help, @albertvillanova! All tests are passing now.']

In [ ]:
# Depending on your internet connection, this can take a few minutes...
issues_with_comments_dataset = issues_dataset.map(
    lambda x: {"comments": get_comments(x["number"])}
)

In [ ]:
issues_with_comments_dataset.to_json("issues-datasets-with-comments.jsonl")

In [20]:
from huggingface_hub import list_datasets

all_datasets = list_datasets(author="huggingface")
len(list(all_datasets))

36

In [21]:
from huggingface_hub import notebook_login

notebook_login()

In [23]:
from huggingface_hub import create_repo

repo_url = create_repo(repo_id="github-issues", repo_type="dataset")
repo_url

RepoUrl('https://huggingface.co/datasets/goosmanlei/github-issues', endpoint='https://huggingface.co', repo_type='dataset', repo_id='goosmanlei/github-issues')

In [27]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj="datasets-issues.jsonl",
    path_in_repo="datasets-issues.jsonl",
    repo_id="goosmanlei/github-issues",
    repo_type="dataset",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/goosmanlei/github-issues/commit/b5360c70f35db5039061deb9ee432ccddcdc7a69', commit_message='Upload datasets-issues.jsonl with huggingface_hub', commit_description='', oid='b5360c70f35db5039061deb9ee432ccddcdc7a69', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/goosmanlei/github-issues', endpoint='https://huggingface.co', repo_type='dataset', repo_id='goosmanlei/github-issues'), pr_revision=None, pr_num=None)

In [29]:
from datasets import load_dataset

issues_dataset = load_dataset(
    "goosmanlei/github-issues",
    split="train",
    chunksize=30 << 20,  # 30MB可以完整推断字段类型
)
issues_dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment'],
    num_rows: 7980
})

In [32]:
issues_dataset.set_format(type="pandas")

In [34]:
issues_dataset['labels'].value_counts()

labels
[{'id': 1935892857, 'node_id': 'MDU6TGFiZWwxOTM1ODkyODU3', 'url': 'https://api.github.com/repos/huggingface/datasets/labels/bug', 'name': 'bug', 'color': 'd73a4a', 'default': True, 'description': 'Something isn't working'}]                                                                                                                                                                                                                                                             657
[{'id': 1935892871, 'node_id': 'MDU6TGFiZWwxOTM1ODkyODcx', 'url': 'https://api.github.com/repos/huggingface/datasets/labels/enhancement', 'name': 'enhancement', 'color': 'a2eeef', 'default': True, 'description': 'New feature or request'}]                                                                                                                                                                                                                                              427
[{'id': 2067376369, 'node_id': 'M